# Topic 23 - Cosine Similarity

Companion notebook for `cosine.py`. Compute cosine similarity by hand on two tiny vectors, see exactly why cosine beats Euclidean distance (magnitude invariance), then walk the same math through numpy into the pgvector operators `<->`, `<=>`, `<#>` - and calibrate a threshold per corpus instead of copy-pasting a folk number.

## Learning objectives

- see the topic's mechanics running offline, cell by cell
- connect each demo cell to the pytest suite that pins it
- walk away with numbers you can quote from memory

## 0. Import the core

In [1]:
from cosine import (
    HAND_COMPUTATION, angle_degrees, best_f1_threshold, calibration_report,
    cosine_distance, cosine_similarity, dot_product, euclidean_distance,
    magnitude, negative_inner_product, np_cosine_similarity_matrix,
    np_rank_by_similarity, sweep_thresholds,
)


## 1. The hand computation: a = (1,2), b = (2,1)

dot = 4; |a| = |b| = sqrt(5); cos = 4/5 = 0.8; theta = 36.87 degrees; cosine distance = 0.2. The tests assert these very numbers.

In [2]:
a, b = (1, 2), (2, 1)
print(f"dot             = {dot_product(a, b)}")
print(f"|a|, |b|        = {magnitude(a):.4f}, {magnitude(b):.4f}")
print(f"cosine          = {cosine_similarity(a, b)}")
print(f"angle           = {angle_degrees(a, b):.4f} degrees")
print(f"cosine distance = {cosine_distance(a, b):.12f}")
assert HAND_COMPUTATION["cosine"] == 0.8
print("\n== the reference table HAND_COMPUTATION ==")
for key, value in HAND_COMPUTATION.items():
    print(f"  {key:15s} {value}")


dot             = 4
|a|, |b|        = 2.2361, 2.2361
cosine          = 0.7999999999999998
angle           = 36.8699 degrees
cosine distance = 0.200000000000

== the reference table HAND_COMPUTATION ==
  a               (1, 2)
  b               (2, 1)
  dot             4
  mag_a           2.23606797749979
  mag_b           2.23606797749979
  cosine          0.8
  angle_degrees   36.86989764584402
  cosine_distance 0.2


## 2. Angles you already know

Same direction = 1, right angle = 0 (unrelated), opposite = -1.

In [3]:
x = (1, 0)
for vec, label in [((1, 1), "45 degrees"), ((0, 1), "90 degrees"), ((-1, 0), "180 degrees")]:
    print(f"  angle(x, {label:12s}) = {angle_degrees(x, vec):7.2f}   cos = {cosine_similarity(x, vec):+.2f}")


  angle(x, 45 degrees  ) =   45.00   cos = +0.71
  angle(x, 90 degrees  ) =   90.00   cos = +0.00
  angle(x, 180 degrees ) =  180.00   cos = -1.00


## 3. Why cosine and not Euclidean: magnitude invariance

Scale a vector by 1000 and its meaning has not changed - cosine agrees, Euclidean explodes. On the unit sphere the two orderings coincide; in general they disagree, and disagree wrongly for text.

In [4]:
q = [1.0, 2.0]
doc = [2.0, 1.0]
scaled = [1000.0 * v for v in q]
print(f"cosine(q, doc)         = {cosine_similarity(q, doc):.4f}")
print(f"cosine(1000*q, doc)    = {cosine_similarity(scaled, doc):.4f}   <- unmoved")
print(f"euclidean(q, doc)      = {euclidean_distance(q, doc):.4f}")
print(f"euclidean(1000*q, doc) = {euclidean_distance(scaled, doc):.4f}   <- 1000x away")


cosine(q, doc)         = 0.8000
cosine(1000*q, doc)    = 0.8000   <- unmoved
euclidean(q, doc)      = 1.4142
euclidean(1000*q, doc) = 2234.2795   <- 1000x away


## 4. numpy: one matrix product replaces a loop

The shape real search takes: a table of embeddings vs a query.

In [5]:
import numpy as np

matrix = np.round(np.random.default_rng(7).normal(size=(6, 4)), 3)  # deterministic
query = np.round(np.random.default_rng(11).normal(size=4), 3)
sims = np_cosine_similarity_matrix(matrix, query)
print("row  cosine_similarity")
for i, s in enumerate(sims):
    print(f"  {i}      {s:+.4f}")
top = np_rank_by_similarity(matrix, query, k=3)
print(f"\ntop-3 rows: {top}")
pure = cosine_similarity(list(query), list(matrix[top[0]]))
assert abs(pure - float(sims[top[0]])) < 1e-12
print("numpy and pure python agree to 1e-12")


row  cosine_similarity
  0      +0.2824
  1      -0.6008
  2      -0.2332
  3      -0.7451
  4      -0.4558
  5      -0.4816

top-3 rows: [0, 2, 4]
numpy and pure python agree to 1e-12


## 5. The pgvector operator mirrors

`<->` L2, `<=>` cosine distance (1 - cos), `<#>` NEGATIVE inner product (Postgres walks ASC order, so 'most similar' must be the smallest value).

In [6]:
q = [1, 2]
rows = {
    "same direction  (2, 4)":  [2, 4],
    "orthogonal      (-2, 1)": [-2, 1],
    "opposite        (-1, -2)": [-1, -2],
}
print(f"{'row':24s} {'<-> L2':>8s} {'<=>':>8s} {'<#>':>8s}")
for label, vec in rows.items():
    l2 = euclidean_distance(q, vec)
    cd = cosine_distance(q, vec)
    nip = negative_inner_product(q, vec)
    print(f"{label:24s} {l2:8.3f} {cd:8.3f} {nip:8.3f}")
print("\nascending `<->` and `<=>` both put 'same direction' first")
print("ascending `<#>` puts it first too: most similar = most negative")


row                        <-> L2      <=>      <#>
same direction  (2, 4)      2.236    0.000  -10.000
orthogonal      (-2, 1)     3.162    1.000    0.000
opposite        (-1, -2)    4.472    2.000    5.000

ascending `<->` and `<=>` both put 'same direction' first
ascending `<#>` puts it first too: most similar = most negative


## 6. Thresholds: no universal number; tune per corpus

Two corpora, one perfect threshold each - and each other's threshold is blind. The sweep prints precision/recall/F1 per cutoff.

In [7]:
noisy_corpus = [(0.93, True), (0.92, True), (0.89, False), (0.88, False)]  # sims bunched high
clean_corpus = [(0.71, True), (0.72, True), (0.30, False), (0.25, False)]  # sims spread low
print("corpus A (noisy separation):")
print(calibration_report(noisy_corpus, [0.915, 0.905, 0.9, 0.89, 0.5]))
print("\ncorpus B (clean separation):")
print(calibration_report(clean_corpus, [0.915, 0.73, 0.6, 0.5, 0.45]))
best_a = best_f1_threshold(noisy_corpus, [0.915, 0.905, 0.9, 0.89, 0.5])
best_b = best_f1_threshold(clean_corpus, [0.915, 0.73, 0.6, 0.5, 0.45])
print(f"\nbest F1 cutoff: corpus A = {best_a.threshold}   corpus B = {best_b.threshold}")
print("a threshold is per-corpus configuration, not a constant\n")


corpus A (noisy separation):
threshold | precision | recall | f1
----------+-----------+--------+----
    0.50 |      0.50 |   1.00 | 0.67
    0.89 |      0.67 |   1.00 | 0.80
    0.90 |      1.00 |   1.00 | 1.00
    0.91 |      1.00 |   1.00 | 1.00
    0.92 |      1.00 |   1.00 | 1.00

corpus B (clean separation):
threshold | precision | recall | f1
----------+-----------+--------+----
    0.45 |      1.00 |   1.00 | 1.00
    0.50 |      1.00 |   1.00 | 1.00
    0.60 |      1.00 |   1.00 | 1.00
    0.73 |      0.00 |   0.00 | 0.00
    0.92 |      0.00 |   0.00 | 0.00

best F1 cutoff: corpus A = 0.9   corpus B = 0.45
a threshold is per-corpus configuration, not a constant



## 7. Run the tests

In [8]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-1500:])
assert "failed" not in result.stdout, "pytest failures!"
print("\noffline tests passed")


..................................                                       [100%]
34 passed in 0.15s


offline tests passed


## 8. What to notice

- cos((1,2),(2,1)) = 0.8 at 36.87 degrees, distance 0.2 - computed before argued
- magnitude invariance is THE reason cosine owns text comparison
- unit vectors make cosine and Euclidean rank identically (the 2-2cos identity)
- `<#>` is negated dot because Postgres index scans walk ascending
- thresholds are corpus-and-model specific; sweep, then pick at the knee
- 34 offline tests pass and the notebook prints 'offline tests passed'